In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.ticker import MaxNLocator
from scipy import ndimage
from scipy.stats import gaussian_kde, ks_2samp, mannwhitneyu
import rasterio
from rasterio.plot import show
from skimage.measure import label, regionprops
from skimage.morphology import dilation, square

In [ ]:
# ============================================================================
# USER CONFIG  ← edit these paths for each scene
# ============================================================================

SCENES = [
    {
        "name":     "set-4",
        "thermal":  "/content/drive/MyDrive/PS_11/stage2prep_holdout/stage2_dataset_extracted/Phase II datasets/Set-4/Thermal/LC09_L2SP_147049_20251121_20251122_02_T1_ST_B10.TIF",
        "gt_mask":  "/content/drive/MyDrive/PS_11/stage2prep_holdout/stage2_dataset_extracted/Phase II datasets/Set-4/Thermal/Groundtruth/groundtruth_thermal.tif",
    },
    # Add more scenes below — comment out any you don't have yet
    # {
    #     "name":    "LC09_141045_20250604",
    #     "thermal": "/content/drive/MyDrive/PS_11/thermal/LC09_...B10.TIF",
    #     "gt_mask": "/content/drive/MyDrive/PS_11/ground_truth/LC09_141045_20250604_gt.tif",
    # },
    # {
    #     "name":    "LC09_150044_20251009",
    #     "thermal": "...",
    #     "gt_mask": "...",
    # },
    # {
    #     "name":    "LC09_147049_20251121",
    #     "thermal": "...",
    #     "gt_mask": "...",
    # },
]

OUTPUT_DIR = "/content/drive/MyDrive/PS_11/stage2_v1_output/thermal_gt/set-4/"   # plots saved here

# ── Landsat Collection-2 ST scaling constants ─────────────────────────────
SCALE_FACTOR = 0.00341802
OFFSET        = 149.0          # result in Kelvin
KELVIN_OFFSET = 273.15         # K → °C
NODATA_DN     = 0

# ── Local window for neighbourhood statistics ─────────────────────────────
WINDOW_SIZE = 11

# ── Colours ───────────────────────────────────────────────────────────────
C_BG  = "#4C72B0"   # blue  – background pixels
C_AN  = "#DD4949"   # red   – anomaly pixels
C_OV  = "#9B59B6"   # purple – overlap region on KDE plot

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def load_scene(thermal_path, gt_path):
    """
    Load thermal band (DN→°C) and ground-truth mask.
    Returns lst_celsius (2D float32), gt_mask (2D bool), rasterio profile.
    """
    with rasterio.open(thermal_path) as src:
        dn   = src.read(1).astype(np.float32)
        profile = src.profile.copy()

    nodata_mask = dn == NODATA_DN
    dn[nodata_mask] = np.nan

    lst_k = dn * SCALE_FACTOR + OFFSET
    lst_c = lst_k - KELVIN_OFFSET
    lst_c[nodata_mask] = np.nan

    with rasterio.open(gt_path) as src:
        gt_raw = src.read(1)

    gt_mask = gt_raw.astype(bool)    # True = anomaly pixel

    return lst_c, gt_mask, profile


def get_local_stats(lst_c, window=WINDOW_SIZE):
    """
    Compute local mean, std, and local RX (z-score) for every valid pixel.
    """
    filled = np.nan_to_num(lst_c, nan=np.nanmean(lst_c))
    local_mean = ndimage.uniform_filter(filled, size=window, mode='reflect')
    mean_sq    = ndimage.uniform_filter(filled**2, size=window, mode='reflect')
    local_var  = np.maximum(mean_sq - local_mean**2, 0)
    local_std  = np.sqrt(local_var)
    local_std[local_std < 1e-6] = 1e-6
    local_rx   = (filled - local_mean) / local_std
    return local_mean, local_std, local_rx


def print_summary(name, lst_c, gt_mask):
    """Print statistical summary to console."""
    valid = ~np.isnan(lst_c)
    bg    = valid & ~gt_mask
    an    = valid &  gt_mask

    total_valid = valid.sum()
    n_an        = an.sum()
    n_bg        = bg.sum()

    print(f"\n{'='*65}")
    print(f"  SCENE: {name}")
    print(f"{'='*65}")
    print(f"  Total pixels      : {lst_c.size:>12,}")
    print(f"  Valid pixels      : {total_valid:>12,}")
    print(f"  Anomaly pixels    : {n_an:>12,}  ({100*n_an/total_valid:.4f}%)")
    print(f"  Background pixels : {n_bg:>12,}  ({100*n_bg/total_valid:.4f}%)")
    print(f"\n  {'Metric':<25} {'Background':>12} {'Anomaly':>12}")
    print(f"  {'-'*49}")
    for label_str, arr in [("Mean (°C)", lst_c),
                             ("Std (°C)",  lst_c),
                             ("Min (°C)",  lst_c),
                             ("Max (°C)",  lst_c),
                             ("Median (°C)", lst_c)]:
        bg_val = (np.nanmean(arr[bg])   if "Mean"   in label_str else
                  np.nanstd(arr[bg])    if "Std"    in label_str else
                  np.nanmin(arr[bg])    if "Min"    in label_str else
                  np.nanmax(arr[bg])    if "Max"    in label_str else
                  np.nanmedian(arr[bg]))
        an_val = (np.nanmean(arr[an])   if "Mean"   in label_str else
                  np.nanstd(arr[an])    if "Std"    in label_str else
                  np.nanmin(arr[an])    if "Min"    in label_str else
                  np.nanmax(arr[an])    if "Max"    in label_str else
                  np.nanmedian(arr[an]))
        print(f"  {label_str:<25} {bg_val:>12.2f} {an_val:>12.2f}")

    # Statistical tests
    bg_vals = lst_c[bg][~np.isnan(lst_c[bg])]
    an_vals = lst_c[an][~np.isnan(lst_c[an])]
    ks_stat, ks_p   = ks_2samp(bg_vals[:50000], an_vals)   # subsample bg for speed
    mw_stat, mw_p   = mannwhitneyu(bg_vals[:50000], an_vals, alternative='two-sided')
    print(f"\n  Kolmogorov-Smirnov test  : stat={ks_stat:.4f},  p={ks_p:.2e}")
    print(f"  Mann-Whitney U test      : stat={mw_stat:.4f},  p={mw_p:.2e}")
    print(f"  → Distributions {'ARE' if ks_p < 0.05 else 'are NOT'} significantly different (α=0.05)")

In [ ]:
# ============================================================================
# PLOT 1 — LST DISTRIBUTION COMPARISON (KDE + Histogram)
# ============================================================================

def plot_lst_distributions(scenes_data, save_path):
    """
    Side-by-side KDE + histogram for background vs anomaly LST
    for every scene in a single figure.
    """
    n = len(scenes_data)
    fig, axes = plt.subplots(2, n, figsize=(6*n, 10),
                              gridspec_kw={'height_ratios': [2, 1]})
    if n == 1:
        axes = axes.reshape(2, 1)

    fig.suptitle("LST Distribution: Background vs Anomaly Pixels",
                 fontsize=15, fontweight='bold', y=1.01)

    for col, sd in enumerate(scenes_data):
        lst_c  = sd['lst_c']
        gt     = sd['gt_mask']
        name   = sd['name']
        valid  = ~np.isnan(lst_c)
        bg_arr = lst_c[valid & ~gt]
        an_arr = lst_c[valid &  gt]

        # ── KDE plot ──────────────────────────────────────────────────────
        ax_kde = axes[0, col]

        # subsample background for KDE speed
        bg_samp = np.random.choice(bg_arr, size=min(100_000, len(bg_arr)),
                                    replace=False)
        kde_bg = gaussian_kde(bg_samp, bw_method='scott')
        kde_an = gaussian_kde(an_arr,  bw_method='scott')

        x_min = min(np.percentile(bg_samp, 0.1), an_arr.min())
        x_max = max(np.percentile(bg_samp, 99.9), an_arr.max())
        xs = np.linspace(x_min, x_max, 800)

        y_bg = kde_bg(xs)
        y_an = kde_an(xs)

        ax_kde.fill_between(xs, y_bg, alpha=0.35, color=C_BG)
        ax_kde.fill_between(xs, y_an, alpha=0.50, color=C_AN)
        ax_kde.plot(xs, y_bg, color=C_BG, lw=2,
                    label=f"Background  n={len(bg_arr):,}")
        ax_kde.plot(xs, y_an, color=C_AN, lw=2,
                    label=f"Anomaly  n={len(an_arr):,}")

        # overlap shading
        y_overlap = np.minimum(y_bg, y_an)
        ax_kde.fill_between(xs, y_overlap, alpha=0.4, color=C_OV,
                             label="Overlap region")

        # vertical lines for means
        ax_kde.axvline(np.mean(bg_samp), color=C_BG, ls='--', lw=1.5,
                       label=f"BG mean={np.mean(bg_samp):.1f}°C")
        ax_kde.axvline(np.mean(an_arr),  color=C_AN, ls='--', lw=1.5,
                       label=f"AN mean={np.mean(an_arr):.1f}°C")

        ax_kde.set_title(name, fontsize=11, fontweight='bold')
        ax_kde.set_xlabel("LST (°C)", fontsize=10)
        ax_kde.set_ylabel("Density", fontsize=10)
        ax_kde.legend(fontsize=8, framealpha=0.85)
        ax_kde.grid(True, alpha=0.3)

        # ── Histogram (anomaly pixels only, zoomed) ───────────────────────
        ax_hist = axes[1, col]
        ax_hist.hist(an_arr, bins=40, color=C_AN, alpha=0.8,
                     edgecolor='white', linewidth=0.5)
        ax_hist.set_title("Anomaly LST Histogram (zoomed)", fontsize=10)
        ax_hist.set_xlabel("LST (°C)", fontsize=10)
        ax_hist.set_ylabel("Count", fontsize=10)
        ax_hist.grid(True, alpha=0.3)

        # annotate percentiles
        for pct in [25, 50, 75]:
            v = np.percentile(an_arr, pct)
            ax_hist.axvline(v, color='black', ls=':', lw=1.2)
            ax_hist.text(v, ax_hist.get_ylim()[1]*0.85,
                         f"P{pct}\n{v:.1f}°C",
                         ha='center', fontsize=7, color='black')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"\n[Saved] {save_path}")
    plt.show()

# ============================================================================
# PLOT 2 — SPATIAL DISTRIBUTION OF ANOMALIES (Scatter + Density)
# ============================================================================

def plot_spatial_distribution(scenes_data, save_path):
    """
    For each scene:
      (a) Full scene LST heatmap with anomaly overlay
      (b) Downsampled scatter of anomaly pixel locations
      (c) Anomaly cluster size distribution
    """
    n = len(scenes_data)
    fig = plt.figure(figsize=(15, 6*n))
    outer = gridspec.GridSpec(n, 3, figure=fig, hspace=0.4, wspace=0.35)

    fig.suptitle("Spatial Distribution of Ground-Truth Anomalies",
                 fontsize=14, fontweight='bold', y=1.01)

    for row, sd in enumerate(scenes_data):
        lst_c  = sd['lst_c']
        gt     = sd['gt_mask']
        name   = sd['name']
        valid  = ~np.isnan(lst_c)

        # ── (a) Downsampled LST heatmap with anomaly overlay ──────────────
        ax_map = fig.add_subplot(outer[row, 0])
        ds = 8   # downsample factor for display
        lst_ds = lst_c[::ds, ::ds]
        gt_ds  = gt[::ds, ::ds]

        im = ax_map.imshow(lst_ds, cmap='inferno', aspect='auto',
                           interpolation='nearest')
        # overlay anomaly pixels in bright green
        an_overlay = np.zeros((*lst_ds.shape, 4), dtype=np.float32)
        an_overlay[gt_ds, :] = [0, 1, 0, 0.85]
        ax_map.imshow(an_overlay, aspect='auto', interpolation='nearest')

        plt.colorbar(im, ax=ax_map, fraction=0.03, pad=0.04,
                     label='LST (°C)')
        ax_map.set_title(f"{name}\nLST heatmap + anomalies (green)",
                         fontsize=9, fontweight='bold')
        ax_map.axis('off')

        # ── (b) Anomaly pixel scatter (full resolution row/col) ───────────
        ax_sc = fig.add_subplot(outer[row, 1])
        an_rows, an_cols = np.where(gt & valid)

        # compute LST per anomaly pixel for colour coding
        an_lst_vals = lst_c[an_rows, an_cols]

        # scatter — colour by LST value
        sc = ax_sc.scatter(an_cols, -an_rows,     # flip Y so north is up
                           c=an_lst_vals,
                           cmap='hot', s=2, alpha=0.6,
                           vmin=np.percentile(an_lst_vals, 5),
                           vmax=np.percentile(an_lst_vals, 95))
        plt.colorbar(sc, ax=ax_sc, fraction=0.03, pad=0.04,
                     label='LST (°C)')
        ax_sc.set_title(f"Anomaly pixel locations\n"
                        f"coloured by LST (n={len(an_rows):,})",
                        fontsize=9, fontweight='bold')
        ax_sc.set_xlabel("Column (pixel)", fontsize=8)
        ax_sc.set_ylabel("Row (pixel, flipped)", fontsize=8)
        ax_sc.grid(True, alpha=0.2)

        # ── (c) Cluster size distribution ─────────────────────────────────
        ax_cl = fig.add_subplot(outer[row, 2])
        labeled_gt = label(gt.astype(int))
        props = regionprops(labeled_gt)
        cluster_sizes = np.array([p.area for p in props])

        if len(cluster_sizes) > 0:
            # log-scale histogram
            log_bins = np.logspace(np.log10(max(1, cluster_sizes.min())),
                                    np.log10(cluster_sizes.max() + 1), 30)
            ax_cl.hist(cluster_sizes, bins=log_bins,
                       color=C_AN, edgecolor='white', alpha=0.85)
            ax_cl.set_xscale('log')
            ax_cl.axvline(np.median(cluster_sizes), color='black',
                          ls='--', lw=1.5,
                          label=f"Median={np.median(cluster_sizes):.0f} px")
            ax_cl.axvline(np.mean(cluster_sizes), color='orange',
                          ls='--', lw=1.5,
                          label=f"Mean={np.mean(cluster_sizes):.0f} px")
            ax_cl.legend(fontsize=8)
            ax_cl.set_title(f"Anomaly cluster size distribution\n"
                             f"({len(cluster_sizes)} clusters, "
                             f"total {cluster_sizes.sum():,} px)",
                             fontsize=9, fontweight='bold')
            ax_cl.set_xlabel("Cluster size (pixels, log scale)", fontsize=8)
            ax_cl.set_ylabel("Count", fontsize=8)
            ax_cl.grid(True, alpha=0.3)

            # print cluster stats
            print(f"\n  [{name}] Cluster statistics:")
            print(f"    Total clusters : {len(cluster_sizes)}")
            print(f"    Sizes — min: {cluster_sizes.min()}, "
                  f"max: {cluster_sizes.max()}, "
                  f"median: {np.median(cluster_sizes):.0f}, "
                  f"mean: {np.mean(cluster_sizes):.1f}")
            for thr in [1, 5, 10, 50, 100]:
                pct = 100 * (cluster_sizes <= thr).sum() / len(cluster_sizes)
                print(f"    % clusters ≤ {thr:>4} px : {pct:.1f}%")
        else:
            ax_cl.text(0.5, 0.5, "No anomaly clusters found",
                       ha='center', va='center', transform=ax_cl.transAxes)

    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"\n[Saved] {save_path}")
    plt.show()

In [ ]:
# ============================================================================
# MAIN
# ============================================================================

def main():
    print("\n" + "="*65)
    print("  LST DISTRIBUTION ANALYSIS PIPELINE")
    print("="*65)

    # ── Load all scenes ───────────────────────────────────────────────────
    scenes_data = []
    for sc in SCENES:
        print(f"\nLoading: {sc['name']} ...")
        lst_c, gt_mask, profile = load_scene(sc['thermal'], sc['gt_mask'])
        local_mean, local_std, local_rx = get_local_stats(lst_c)
        scenes_data.append({
            'name':       sc['name'],
            'lst_c':      lst_c,
            'gt_mask':    gt_mask,
            'local_mean': local_mean,
            'local_std':  local_std,
            'local_rx':   local_rx,
            'profile':    profile,
        })
        print_summary(sc['name'], lst_c, gt_mask)

    # ── Generate all plots ────────────────────────────────────────────────
    print("\n\nGenerating plots...")

    plot_lst_distributions(
        scenes_data,
        os.path.join(OUTPUT_DIR, "plot1_lst_distributions.png"))

    plot_spatial_distribution(
        scenes_data,
        os.path.join(OUTPUT_DIR, "plot3_spatial_distribution.png"))

    print("\n" + "="*65)
    print("  ANALYSIS COMPLETE")
    print(f"  All plots saved to: {OUTPUT_DIR}")
    print("="*65)


if __name__ == "__main__":
    main()